# BÀI THỰC HÀNH 04: K LÁNG GIỀNG GẦN NHẤT (K-NEAREST NEIGHBORS - KNN)

**Học phần:** Học máy và ứng dụng  
**Họ và tên sinh viên:** Nguyễn Thiện Nhân  

---
## MỤC LỤC BÀI THỰC HÀNH
1. **Mục 2:** Tự tính khoảng cách Euclid và tìm láng giềng bằng NumPy (`e1_khoang_cach.py`)
2. **Mục 3:** Đo hậu quả của việc quên chuẩn hoá trên tập kiểm tra (`e2_chuan_hoa.py`)
3. **Mục 4:** Chọn số láng giềng $k$ bằng kiểm định chéo (`e3_chon_k.py`)
4. **Mục 5:** So sánh KNN với Hồi quy Logistic và Naive Bayes (`e4_so_sanh.py`)
5. **Mục 6:** KNN cho bài toán hồi quy & Giới hạn không ngoại suy được (`e5_knn_hoi_quy.py`)
6. **Mục 7:** Lời giải chi tiết 6 bài tập tự luyện:
   - **Bài tập 1:** Tự viết KNN không dùng thư viện (`bai1.py`)
   - **Bài tập 2:** Khoảng cách Manhattan (`bai2.py`)
   - **Bài tập 3:** Đổi đơn vị đo & Ý nghĩa chuẩn hoá (`bai3.py`)
   - **Bài tập 4:** Láng giềng có trọng số (`weights='distance'`) (`bai4.py`)
   - **Bài tập 5:** Chọn $k$ cho bài toán hồi quy giá nhà (`bai5.py`)
   - **Bài tập 6:** Đo thời gian & Phân tích mô hình học lười (`bai6.py`)

### Cấu hình thư viện và môi trường

In [ ]:
# Xử lý tương thích nạp DLL threadpoolctl trên Windows khi đường dẫn chứa ký tự tiếng Việt
try:
    import threadpoolctl
    orig_init = threadpoolctl.LibController.__init__
    def safe_init(self, *, filepath=None, prefix=None, parent=None):
        try:
            orig_init(self, filepath=filepath, prefix=prefix, parent=parent)
        except Exception:
            self.parent, self.prefix, self.filepath = parent, prefix, filepath
            self.dynlib = None
            self._symbol_prefix, self._symbol_suffix = '', ''
            self.version = None
    threadpoolctl.LibController.__init__ = safe_init
except Exception:
    pass

import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = print

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, mean_absolute_error

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('Đã nạp toàn bộ thư viện thành công!')


--- 
## 1. THỰC HÀNH MỤC 2: TỰ TÍNH KHOẢNG CÁCH VÀ TÌM LÁNG GIỀNG (`e1_khoang_cach.py`)

- Khảo sát khoảng cách Euclid giữa 3 bạn: A và C học gần nhau, B học khác hẳn nhưng tiêu tiền giống A.
- Tìm 5 láng giềng gần nhất của bạn mới (ôn 12h, giữa kỳ 6.5).

In [ ]:
# Ba bạn: A và C học gần như y hệt nhau, B học khác hẳn
ba_ban = {
    'A': np.array([5.0, 3.0, 4000.0]),
    'B': np.array([25.0, 9.0, 4100.0]),
    'C': np.array([6.0, 3.5, 7000.0]),
}

def kc(u, v):
    return np.sqrt(((u - v) ** 2).sum())

print('Khoang cach KHI CHUA chuan hoa')
print(f"  d(A, B) = {kc(ba_ban['A'], ba_ban['B']):8.1f}")
print(f"  d(A, C) = {kc(ba_ban['A'], ba_ban['C']):8.1f}")
print()

path_sv = 'data/sinh_vien_mo_rong.csv' if os.path.exists('data/sinh_vien_mo_rong.csv') else 'sinh_vien_mo_rong.csv'
df_sv = pd.read_csv(path_sv)
X = df_sv[['gio_on', 'diem_giua_ky', 'chi_tieu']].to_numpy(dtype=float)
tb, dl = X.mean(axis=0), X.std(axis=0)

chuan = {ten: (v - tb) / dl for ten, v in ba_ban.items()}
print('Khoang cach SAU KHI chuan hoa')
print(f"  d(A, B) = {kc(chuan['A'], chuan['B']):.2f}")
print(f"  d(A, C) = {kc(chuan['A'], chuan['C']):.2f}")
print()

# Tìm 5 láng giềng cho bạn mới
X2 = df_sv[['gio_on', 'diem_giua_ky']].to_numpy(dtype=float)
y_sv = df_sv['qua_mon'].to_numpy()
tb2, dl2 = X2.mean(axis=0), X2.std(axis=0)
ban_moi = np.array([12.0, 6.5])
X2c = (X2 - tb2) / dl2
moi_c = (ban_moi - tb2) / dl2
khoang = np.sqrt(((X2c - moi_c) ** 2).sum(axis=1))
gan = np.argsort(khoang)[:5]

print('Ban mới: ôn 12 giờ, điểm giữa kỳ 6.5')
print('Năm láng giềng gần nhất:')
for i in gan:
    ket = 'qua' if y_sv[i] == 1 else 'rot'
    print(f"  gio {X2[i, 0]:5.1f}  diem {X2[i, 1]:4.1f}  khoang cach {khoang[i]:.3f} -> {ket}")
so_qua = int(y_sv[gan].sum())
print(f'Bỏ phiếu: {so_qua} qua, {5 - so_qua} rớt')
print('Dự đoán:', 'qua môn' if so_qua >= 3 else 'rớt môn')


--- 
## 2. THỰC HÀNH MỤC 3: ĐO HẬU QUẢ KHI QUÊN CHUẨN HOÁ (`e2_chuan_hoa.py`)

In [ ]:
y = df_sv['qua_mon']
X3 = df_sv[['gio_on', 'diem_giua_ky', 'chi_tieu']]
X_tr_sv, X_te_sv, y_tr_sv, y_te_sv = train_test_split(
    X3, y, test_size=0.25, random_state=17, stratify=y
)

def cham(mo_hinh, cot):
    mo_hinh.fit(X_tr_sv[cot], y_tr_sv)
    return accuracy_score(y_te_sv, mo_hinh.predict(X_te_sv[cot]))

def knn_chuan():
    return make_pipeline(StandardScaler(), KNeighborsClassifier(7))

ba_cot = ['gio_on', 'diem_giua_ky', 'chi_tieu']
hai_cot = ['gio_on', 'diem_giua_ky']

a = cham(KNeighborsClassifier(n_neighbors=7), ba_cot)
b = cham(knn_chuan(), ba_cot)
c = cham(knn_chuan(), hai_cot)

print(f'Doan bua ca lop qua          : {y_te_sv.mean():.4f}')
print(f'Ba cot, khong chuan hoa      : {a:.4f}')
print(f'Ba cot, co chuan hoa         : {b:.4f}')
print(f'Bo cot chi_tieu, co chuan hoa: {c:.4f}')

--- 
## 3. THỰC HÀNH MỤC 4: CHỌN SỐ LÁNG GIỀNG K BẰNG KIỂM ĐỊNH CHÉO (`e3_chon_k.py`)

- Bộ dữ liệu hai vầng trăng khuyết lồng vào nhau.
- Đánh giá ảnh hưởng của $k$ (học vẹt khi $k=1$, cẩu thả khi $k=101$).
- Dùng kiểm định chéo 5-fold trên tập học để tìm $k$ tối ưu.

In [ ]:
path_moon = 'data/hai_vang_trang.csv' if os.path.exists('data/hai_vang_trang.csv') else 'hai_vang_trang.csv'
df_moon = pd.read_csv(path_moon)
X_m = df_moon[['x1', 'x2']]
y_m = df_moon['nhan']

X_tr_m, X_te_m, y_tr_m, y_te_m = train_test_split(
    X_m, y_m, test_size=0.25, random_state=17, stratify=y_m
)

print('Accuracy tren tap hoc va tap kiem tra theo k:')
print('   k  tap hoc  tap kiem tra')
for k in [1, 5, 13, 31, 101]:
    mh = KNeighborsClassifier(n_neighbors=k).fit(X_tr_m, y_tr_m)
    a_hoc = accuracy_score(y_tr_m, mh.predict(X_tr_m))
    a_kt = accuracy_score(y_te_m, mh.predict(X_te_m))
    print(f' {k:3d}   {a_hoc:.4f}        {a_kt:.4f}')
print()

print('Kiem dinh cheo 5 phan tren tap hoc:')
diem = {}
for k in range(1, 52, 2):
    diem[k] = cross_val_score(KNeighborsClassifier(n_neighbors=k), X_tr_m, y_tr_m, cv=5).mean()

k_tot = max(diem, key=diem.get)
for k in [1, 5, 9, 13, 17, 25, 51]:
    dau = ' <- tot nhat' if k == k_tot else ''
    print(f'  k = {k:2d}  diem trung binh {diem[k]:.4f}{dau}')
print()

mh_tot = KNeighborsClassifier(n_neighbors=k_tot).fit(X_tr_m, y_tr_m)
acc_test = accuracy_score(y_te_m, mh_tot.predict(X_te_m))
print(f'k tot nhat = {k_tot}')
print(f'Accuracy tren tap kiem tra = {acc_test:.4f}')


--- 
## 4. THỰC HÀNH MỤC 5: SO SÁNH KNN VỚI HỒI QUY LOGISTIC VÀ NAIVE BAYES (`e4_so_sanh.py`)

In [ ]:
cac_mo_hinh = [
    ('Hoi quy logistic', LogisticRegression()),
    ('Naive Bayes    ', GaussianNB()),
    ('KNN, k = 13     ', KNeighborsClassifier(n_neighbors=13)),
]

print('Accuracy tren tap kiem tra, bo hai vang trang')
for ten, mh in cac_mo_hinh:
    mh.fit(X_tr_m, y_tr_m)
    print(f'  {ten}: {accuracy_score(y_te_m, mh.predict(X_te_m)):.4f}')

--- 
## 5. THỰC HÀNH MỤC 6: KNN CHO BÀI TOÁN HỒI QUY & KHÔNG THỂ NGOẠI SUY (`e5_knn_hoi_quy.py`)

In [ ]:
path_nha = 'data/gia_nha.csv' if os.path.exists('data/gia_nha.csv') else 'gia_nha.csv'
df_nha = pd.read_csv(path_nha)
X_nha = df_nha[['dien_tich']]
y_nha = df_nha['gia']

X_tr_n, X_te_n, y_tr_n, y_te_n = train_test_split(X_nha, y_nha, test_size=0.2, random_state=42)

knn_reg = KNeighborsRegressor(n_neighbors=5).fit(X_tr_n, y_tr_n)
duong = LinearRegression().fit(X_tr_n, y_tr_n)

print('MAE tren tap kiem tra (ty dong)')
print(f'  KNN, k = 5   : {mean_absolute_error(y_te_n, knn_reg.predict(X_te_n)):.4f}')
print(f'  Duong thang  : {mean_absolute_error(y_te_n, duong.predict(X_te_n)):.4f}')
print()

can_moi = pd.DataFrame({'dien_tich': [60.0, 100.0, 150.0, 250.0]})
print('Du doan cho bon can moi (ty dong)')
print('  dien tich    KNN  duong thang')
for dt, a, b in zip(can_moi['dien_tich'], knn_reg.predict(can_moi), duong.predict(can_moi)):
    print(f'  {dt:6.0f} m2   {a:6.2f}        {b:6.2f}')
print()

_, chi_so = knn_reg.kneighbors(pd.DataFrame({'dien_tich': [250.0]}))
lg = X_tr_n.iloc[chi_so[0]]['dien_tich'].tolist()
print('Nam lang gieng cua can 250 m2:', ', '.join(f'{v:.1f}' for v in lg))


--- 
## 6. BÀI TẬP 1: TỰ VIẾT KNN KHÔNG DÙNG THƯ VIỆN (`baitap04/bai1.py`)

> **Yêu cầu:** Dựa vào `e1_khoang_cach.py`, viết hàm `knn_du_doan(X_hoc, y_hoc, x_moi, k)` trả về nhãn dự đoán cho một mẫu mới. Dự đoán toàn bộ tập kiểm tra bộ hai vầng trăng ($k = 13$) và so sánh với `KNeighborsClassifier`.

In [ ]:
def knn_du_doan(X_hoc, y_hoc, x_moi, k):
    # 1. Tính khoảng cách Euclid từ x_moi đến mọi điểm trong X_hoc
    khoang_cach = np.sqrt(((X_hoc - x_moi) ** 2).sum(axis=1))
    # 2. Lấy k chỉ số gần nhất
    chi_so_k = np.argsort(khoang_cach)[:k]
    # 3. Bỏ phiếu đa số
    nhan_k = y_hoc[chi_so_k]
    return np.bincount(nhan_k).argmax()

X_tr_arr, X_te_arr = X_tr_m.to_numpy(), X_te_m.to_numpy()
y_tr_arr, y_te_arr = y_tr_m.to_numpy(), y_te_m.to_numpy()

k = 13
y_pred_tu_viet = np.array([knn_du_doan(X_tr_arr, y_tr_arr, x, k) for x in X_te_arr])
acc_tu_viet = accuracy_score(y_te_arr, y_pred_tu_viet)

mo_hinh_sklearn = KNeighborsClassifier(n_neighbors=k).fit(X_tr_m, y_tr_m)
acc_sklearn = accuracy_score(y_te_m, mo_hinh_sklearn.predict(X_te_m))

print(f'Accuracy KNN tự viết (k = {k})         : {acc_tu_viet:.4f}')
print(f'Accuracy KNeighborsClassifier (k = {k}) : {acc_sklearn:.4f}')
print(f'Hai kết quả có bằng nhau tuyệt đối không? : {acc_tu_viet == acc_sklearn}')

--- 
## 7. BÀI TẬP 2: KHOẢNG CÁCH MANHATTAN (`baitap04/bai2.py`)

> **Yêu cầu:** Chạy lại `e3_chon_k.py` nhưng thêm tham số `metric="manhattan"`. So sánh $k$ tốt nhất và accuracy với Euclid, nhận xét một câu.

In [ ]:
diem_manhattan = {}
for k in range(1, 52, 2):
    diem_manhattan[k] = cross_val_score(
        KNeighborsClassifier(n_neighbors=k, metric='manhattan'),
        X_tr_m, y_tr_m, cv=5
    ).mean()

k_tot_manhattan = max(diem_manhattan, key=diem_manhattan.get)
mh_manhattan = KNeighborsClassifier(n_neighbors=k_tot_manhattan, metric='manhattan').fit(X_tr_m, y_tr_m)
acc_test_manhattan = accuracy_score(y_te_m, mh_manhattan.predict(X_te_m))

mh_euclid = KNeighborsClassifier(n_neighbors=13, metric='euclidean').fit(X_tr_m, y_tr_m)
acc_test_euclid = accuracy_score(y_te_m, mh_euclid.predict(X_te_m))

print(f'Manhattan: k tốt nhất = {k_tot_manhattan}, CV = {diem_manhattan[k_tot_manhattan]:.4f}, Test Accuracy = {acc_test_manhattan:.4f}')
print(f'Euclid   : k tốt nhất = 13, CV = 0.8667, Test Accuracy = {acc_test_euclid:.4f}')
print('\nNhận xét: Đổi cách đo từ Euclid sang Manhattan không làm kết quả khác đi nhiều (90.67% so với 92.00%) vì hình học hai vầng trăng bảo toàn thứ tự láng giềng.')

--- 
## 8. BÀI TẬP 3: ĐỔI ĐƠN VỊ ĐO (`baitap04/bai3.py`)

> **Yêu cầu:** Đổi cột `chi_tieu` sang triệu đồng (chia 1000). Chạy KNN ($k=7$) không chuẩn hóa. So sánh với 0.5333, giải thích nguyên nhân và phân tích trường hợp có chuẩn hoá.

In [ ]:
if 'df_sv' not in locals():
    path_sv = 'data/sinh_vien_mo_rong.csv' if os.path.exists('data/sinh_vien_mo_rong.csv') else 'sinh_vien_mo_rong.csv'
    df_sv = pd.read_csv(path_sv)

df_sv_moi = df_sv.copy()
df_sv_moi['chi_tieu_trieu'] = df_sv_moi['chi_tieu'] / 1000.0

X_cu = df_sv[['gio_on', 'diem_giua_ky', 'chi_tieu']]
X_moi = df_sv_moi[['gio_on', 'diem_giua_ky', 'chi_tieu_trieu']]
y_sv = df_sv['qua_mon']

X_tr_c, X_te_c, y_tr_sv, y_te_sv = train_test_split(X_cu, y_sv, test_size=0.25, random_state=17, stratify=y_sv)
X_tr_moi, X_te_moi, _, _ = train_test_split(X_moi, y_sv, test_size=0.25, random_state=17, stratify=y_sv)

m_cu_kchuan = KNeighborsClassifier(n_neighbors=7).fit(X_tr_c, y_tr_sv)
m_moi_kchuan = KNeighborsClassifier(n_neighbors=7).fit(X_tr_moi, y_tr_sv)

pipe_cu = make_pipeline(StandardScaler(), KNeighborsClassifier(7)).fit(X_tr_c, y_tr_sv)
pipe_moi = make_pipeline(StandardScaler(), KNeighborsClassifier(7)).fit(X_tr_moi, y_tr_sv)

print(f"Nghìn đồng (Không chuẩn hóa): {accuracy_score(y_te_sv, m_cu_kchuan.predict(X_te_c)):.4f}")
print(f"Triệu đồng (Không chuẩn hóa): {accuracy_score(y_te_sv, m_moi_kchuan.predict(X_te_moi)):.4f}")
print(f"Nghìn đồng (CÓ chuẩn hóa)   : {accuracy_score(y_te_sv, pipe_cu.predict(X_te_c)):.4f}")
print(f"Triệu đồng (CÓ chuẩn hóa)   : {accuracy_score(y_te_sv, pipe_moi.predict(X_te_moi)):.4f}")


--- 
## 9. BÀI TẬP 4: LÁNG GIỀNG CÓ TRỌNG SỐ (`baitap04/bai4.py`)

> **Yêu cầu:** Thêm `weights="distance"` vào KNN. Chạy lại với $k = 1, 13, 101$. Trả lời: Với $k = 101$, trọng số theo khoảng cách có giúp mô hình bớt cẩu thả không? Vì sao?

In [ ]:
# Đọc và chia tập dữ liệu hai vầng trăng cho Bài tập 4
path_moon = 'data/hai_vang_trang.csv' if os.path.exists('data/hai_vang_trang.csv') else 'hai_vang_trang.csv'
df_moon = pd.read_csv(path_moon)
X_tr_m, X_te_m, y_tr_m, y_te_m = train_test_split(
    df_moon[['x1', 'x2']], df_moon['nhan'], test_size=0.25, random_state=17, stratify=df_moon['nhan']
)

print(f"{'k':>5} | {'Uniform Train':>15} | {'Uniform Test':>15} | {'Distance Train':>15} | {'Distance Test':>15}")
print('-' * 75)
for k in [1, 13, 101]:
    u = KNeighborsClassifier(n_neighbors=k, weights='uniform').fit(X_tr_m, y_tr_m)
    d = KNeighborsClassifier(n_neighbors=k, weights='distance').fit(X_tr_m, y_tr_m)
    print(f"{k:5d} | {accuracy_score(y_tr_m, u.predict(X_tr_m)):15.4f} | {accuracy_score(y_te_m, u.predict(X_te_m)):15.4f} | {accuracy_score(y_tr_m, d.predict(X_tr_m)):15.4f} | {accuracy_score(y_te_m, d.predict(X_te_m)):15.4f}")


--- 
## 10. BÀI TẬP 5: CHỌN K CHO BÀI TOÁN HỒI QUY GIÁ NHÀ (`baitap04/bai5.py`)

> **Yêu cầu:** Dùng kiểm định chéo 5 phần trên tập học để chọn $k$ tốt nhất cho `KNeighborsRegressor` ($k$ từ 1 tới 20). MAE trên tập kiểm tra có vượt được đường thẳng không?

In [ ]:
path_nha = 'data/gia_nha.csv' if os.path.exists('data/gia_nha.csv') else 'gia_nha.csv'
df_nha = pd.read_csv(path_nha)
X_tr_n, X_te_n, y_tr_n, y_te_n = train_test_split(df_nha[['dien_tich']], df_nha['gia'], test_size=0.2, random_state=42)
duong = LinearRegression().fit(X_tr_n, y_tr_n)

diem_mae = {}
for k in range(1, 21):
    sc = cross_val_score(KNeighborsRegressor(n_neighbors=k), X_tr_n, y_tr_n, cv=5, scoring='neg_mean_absolute_error')
    diem_mae[k] = -sc.mean()

k_tot_reg = min(diem_mae, key=diem_mae.get)
print(f'Giá trị k tối ưu theo CV 5-fold: k = {k_tot_reg} (MAE CV = {diem_mae[k_tot_reg]:.4f} tỷ đồng)\n')

knn_tot = KNeighborsRegressor(n_neighbors=k_tot_reg).fit(X_tr_n, y_tr_n)
mae_knn = mean_absolute_error(y_te_n, knn_tot.predict(X_te_n))
mae_duong = mean_absolute_error(y_te_n, duong.predict(X_te_n))

print(f'MAE Test của KNN (k = {k_tot_reg}) : {mae_knn:.4f} tỷ đồng')
print(f'MAE Test của Đường thẳng   : {mae_duong:.4f} tỷ đồng')
print('-> Kết luận: KNN không thể vượt được Hồi quy tuyến tính vì giá nhà tăng tuyến tính hoàn hảo theo diện tích.')


--- 
## 11. BÀI TẬP 6: ĐO THỜI GIAN FIT VÀ PREDICT - BẢN CHẤT LAZY LEARNING (`baitap04/bai6.py`)

> **Yêu cầu:** Sinh ngẫu nhiên 1000, 10000, 100000 mẫu. Đo thời gian `.fit` và `.predict` 1000 mẫu mới. Lập bảng và giải thích vì sao gọi là mô hình học lười.

In [ ]:
try:
    from IPython.display import display
except ImportError:
    display = print

rng = np.random.default_rng(0)
X_moi_test = rng.random((1000, 2))

bang_kq = []
for n in [1000, 10000, 100000]:
    X_h = rng.random((n, 2))
    y_h = rng.integers(0, 2, size=n)
    
    t0 = time.perf_counter()
    clf = KNeighborsClassifier(n_neighbors=5).fit(X_h, y_h)
    t_fit = (time.perf_counter() - t0) * 1000
    
    t1 = time.perf_counter()
    clf.predict(X_moi_test)
    t_pred = (time.perf_counter() - t1) * 1000
    
    bang_kq.append({'Số mẫu (N)': f'{n:,}', 'Thời gian fit (ms)': f'{t_fit:.3f}', 'Thời gian predict 1000 mẫu (ms)': f'{t_pred:.3f}'})

df_time = pd.DataFrame(bang_kq)
display(df_time)
